# PHISTO Extended Benchmark Through MicrobioLink 2.1

This notebook is the interactive entry point for the broad PHISTO bacteria-human benchmark bundled in this branch. It is designed to be read top to bottom and focuses on the upstream interaction prediction layer only.

It shows how to:

1. rebuild the unique PHISTO true-positive panel from the raw export,
2. export accession lists for the benchmark proteins,
3. optionally redownload FASTA and Pfam annotations with the new packaged UniProt helpers,
4. preview the forward DMI, reverse DMI, and DDI output schemas from the packaged API,
5. rerun the broad PHISTO benchmark through the packaged API into `../10_reproduced_run/`.

Operational notes:

- The reference benchmark outputs already shipped with this branch live in `../05_results/`.
- Any rerun from this notebook writes into `../10_reproduced_run/` so the documented reference bundle remains unchanged.
- Reverse DMI on the full PHISTO panel is large. The support code batches the packaged API calls to keep memory use practical.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

from microbiolink.download_protein_domains import download_protein_list_with_fields
from microbiolink.get_protein_fasta import fetch_fasta_sequences
from microbiolink_api import load_default_ddi_resource_bundle, load_default_dmi_resource_bundle

search_root = Path.cwd().resolve()
benchmark_root = None

for candidate in [search_root, *search_root.parents]:
    direct_candidate = candidate / 'benchmarks' / 'phisto_extended'
    local_candidate = candidate / 'phisto_extended'

    for possible_root in [direct_candidate, local_candidate]:
        if (possible_root / '06_scripts' / 'package_benchmark_workflow.py').exists():
            benchmark_root = possible_root
            break

    if benchmark_root is not None:
        break

if benchmark_root is None:
    raise FileNotFoundError('Could not locate benchmarks/phisto_extended from the current working directory.')

sys.path.insert(0, str(benchmark_root / '06_scripts'))

import package_benchmark_workflow as workflow

paths = workflow.resolve_benchmark_input_paths(benchmark_root)
reproduced_root = benchmark_root / '10_reproduced_run'

print(f'Benchmark root: {benchmark_root}')
print(f'Documented results: {benchmark_root / "05_results"}')
print(f'Regenerated outputs: {reproduced_root}')
print(f'FASTA helper: {fetch_fasta_sequences.__name__}')
print(f'Domain helper: {download_protein_list_with_fields.__name__}')

## Step 1: Rebuild the Broad PHISTO True-Positive Panel

The documented benchmark ships the already deduplicated PHISTO panel in `05_results/phisto_unique_pairs.tsv`, but the first thing we do here is rebuild it directly from the raw PHISTO export so the preprocessing remains transparent.

In [ ]:
raw_phisto = workflow.read_phisto_export(paths.phisto_csv)
unique_pairs = workflow.build_unique_pair_panel(paths.phisto_csv)

panel_summary = pd.DataFrame(
    [
        {'metric': 'raw_phisto_rows', 'value': len(raw_phisto)},
        {'metric': 'unique_pairs', 'value': len(unique_pairs)},
        {'metric': 'unique_bacterial_accessions', 'value': unique_pairs['pathogen_accession'].nunique()},
        {'metric': 'unique_human_accessions', 'value': unique_pairs['human_accession'].nunique()},
    ],
)

display(panel_summary)
display(unique_pairs.head())

## Step 2: Export the Benchmark Accession Lists

The package helpers work from accession lists. This cell writes the one-column bacterial and human accession files that are used by the optional download step and by any external reruns.

In [ ]:
accession_paths = workflow.write_accession_lists(
    unique_pairs = unique_pairs,
    output_dir = reproduced_root / '01_accessions',
)

pd.DataFrame(
    {
        'logical_name': list(accession_paths.keys()),
        'path': [str(path) for path in accession_paths.values()],
    },
)


## Step 3: Optionally Redownload FASTA and Pfam Inputs Through the Package

The documented benchmark already includes the prepared FASTA and Pfam tables in `04_benchmark_inputs/`, so this step is optional. If you set `RUN_NETWORK_DOWNLOADS = True`, the notebook will regenerate those inputs through the new generic package helpers:

- `microbiolink.get_protein_fasta.fetch_fasta_sequences`
- `microbiolink.download_protein_domains.download_protein_list_with_fields`

The regenerated files are written into `../10_reproduced_run/`.

In [ ]:
RUN_NETWORK_DOWNLOADS = False

if RUN_NETWORK_DOWNLOADS:
    regenerated_inputs = workflow.download_panel_inputs(
        unique_pairs = unique_pairs,
        output_dir = reproduced_root,
    )
    display(
        pd.DataFrame(
            {
                'logical_name': list(regenerated_inputs.keys()),
                'path': [str(path) for path in regenerated_inputs.values()],
            },
        ),
    )
else:
    regenerated_inputs = {
        'bacterial_fasta': paths.bacterial_fasta,
        'human_fasta': paths.human_fasta,
        'bacterial_domains': paths.bacterial_domains,
        'human_domains': paths.human_domains,
    }
    print('Network download skipped. Reusing the documented benchmark inputs from 04_benchmark_inputs/.')

## Step 4: Inspect the Packaged Resource Bundles and Prediction Schemas

Before launching the full benchmark, it is useful to inspect the packaged resource sizes and the tabular schemas returned by the new forward DMI, reverse DMI, and DDI APIs.

In [ ]:
dmi_bundle = load_default_dmi_resource_bundle()
ddi_bundle = load_default_ddi_resource_bundle()

resource_overview = pd.DataFrame(
    [
        {
            'resource_bundle': 'default_dmi',
            'rule_count': len(dmi_bundle.motif_domains),
            'regex_count': len(dmi_bundle.elm_regex),
        },
        {
            'resource_bundle': 'default_ddi',
            'rule_count': len(ddi_bundle.pfam_pairs),
            'regex_count': None,
        },
    ],
)

display(resource_overview)

prediction_previews = workflow.preview_prediction_shapes(benchmark_root)

for predictor_name, preview_frame in prediction_previews.items():
    print(predictor_name)
    display(preview_frame)


## Step 5: Run the Broad PHISTO Benchmark Through the Packaged API

The support module wraps the package calls and batches forward and reverse DMI so the full PHISTO panel can be rerun into `../10_reproduced_run/05_results/`.

By default the notebook simply loads the documented reference results already shipped with the branch. Set `RUN_FULL_PACKAGE_BENCHMARK = True` when you want to regenerate the benchmark through the package.

In [ ]:
RUN_FULL_PACKAGE_BENCHMARK = False
FORWARD_BATCH_SIZE = 100
REVERSE_BATCH_SIZE = 25

if RUN_FULL_PACKAGE_BENCHMARK:
    package_results = workflow.run_broad_phisto_benchmark_from_bundle(
        benchmark_root = benchmark_root,
        output_dir = reproduced_root,
        forward_batch_size = FORWARD_BATCH_SIZE,
        reverse_batch_size = REVERSE_BATCH_SIZE,
    )
    print('Regenerated package-based outputs under', reproduced_root)
else:
    package_results = workflow.load_packaged_prediction_tables(benchmark_root)
    print('Loaded the documented reference outputs from 05_results/.')
    print('Set RUN_FULL_PACKAGE_BENCHMARK = True to regenerate them into 10_reproduced_run/.')

display(package_results['approach_summary'])
display(package_results['approach_union_summary'])

## Step 6: Compare the Current Table to the Documented Reference Bundle

This final comparison is useful after a full rerun. If the benchmark was not rerun in this session, the two tables will be identical because the notebook is loading the documented reference outputs.

In [ ]:
documented_results = workflow.load_packaged_prediction_tables(benchmark_root)

comparison_columns = [
    'approach',
    'resource_name',
    'true_positives_recovered',
    'recall_total',
    'recall_on_analyzable_pairs',
    'precision_unique_pairs',
]

comparison = documented_results['approach_summary'][comparison_columns].merge(
    package_results['approach_summary'][comparison_columns],
    on = ['approach', 'resource_name'],
    suffixes = ('_documented', '_current'),
)

display(comparison)
print('Notebook support module:', benchmark_root / '06_scripts' / 'package_benchmark_workflow.py')
print('Interactive notebook:', benchmark_root / '09_notebooks' / 'phisto_extended_package_walkthrough.ipynb')